# dataset_1

## 1. 计算谓词A 和 谓词B的独立性

In [ ]:
import pandas as pd

# 读取评论数据和推文数据
# root = 'D:/softwareResource/datasets/workload4/'
# root = 'D:/softwareResource/datasets/IOGS/many_predicates/independent/dataset_1/'
# root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/valid_data/'
root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'

# comments_df = pd.read_csv(root + 'comments_sorted_valid_4.csv')
# comments_df = pd.read_csv(root + 'comments_valid.csv')
# posts_df = pd.read_csv(root + 'posts_valid.csv')

comments_df = pd.read_csv(root + 'comments.csv')
posts_df = pd.read_csv(root + 'posts.csv')


# 1. 计算 P(A) - 评论的关联帖子满足 M7 的比例
# 获取满足 M7 的帖子 (即 post 中 nli_label == 'entailment')
posts_entailment = posts_df[posts_df['oracle_label'] == 'deepseek_yes']
valid_posts = posts_entailment['id:ID'].values
print(len(valid_posts) / len(posts_df))

# 过滤评论，找出其关联帖子满足 M7 的评论
comments_valid_post = comments_df[comments_df['post'].isin(valid_posts)]
P_A = len(comments_valid_post) / len(comments_df)


# 2. 计算 P(B) - 评论情感为 "agree" 的比例
P_B = len(comments_df[comments_df['nli_label'] == 'bart_entailment']) / len(comments_df)


# 3. 计算 P(A ∩ B) - 评论同时满足关联帖子的 M7 条件和自身情感为 "agree"
comments_valid_both = comments_valid_post[comments_valid_post['nli_label'] == 'bart_entailment']
P_A_and_B = len(comments_valid_both) / len(comments_df)

print('A count:'+ str(len(valid_posts)))
print('B count:'+ str(len(comments_df[comments_df['nli_label'] == 'bart_entailment'])))
print('A+B count:'+ str(len(comments_valid_both)))

# 输出结果
print(f"P(A): {len(valid_posts) / len(posts_df)}")
print(f"P(B): {P_B}")
print(f"P(A ∩ B): {P_A_and_B}")
    
print(len(comments_df))
print(len(posts_df))
print(len(valid_posts))
print(len(comments_valid_post))
print(len(comments_df[comments_df['nli_label'] == 'bart_entailment']) )
print(len(comments_valid_both))

0.03178206583427923
P(A): 0.026618237016155264
P(B): 0.13298276049007915
P(A ∩ B): 0.004336983627886805

## 2.调整这个数据集，使P(A ∩ B) = P(A)*P(B)成立
2.1 计算增加或删除多少评论post才能使P(A ∩ B) = P(A)*P(B)成立

In [ ]:
# 下面这个是计算需要增加多少post才能使P(A ∩ B) = P(A)*P(B)成立
def calculate_additional_posts(p_b, p_a_and_b, current_posts ,total_posts):
    # 计算当前的 P(A) * P(B)
    p_a_exception = p_a_and_b / p_b
    # 计算需要增加的帖子数量
    additional_posts = total_posts * p_a_exception - current_posts
    print(f"需要增加 {additional_posts} 条帖子才能使 P(A ∩ B) = P(A) * P(B) 成立。")
    return additional_posts
calculate_additional_posts(P_B, P_A_and_B, len(valid_posts), len(posts_df))

# 下面这个是计算需要删除多少post才能使P(A ∩ B) = P(A)*P(B)成立
def calculate_minus_posts(p_b, p_a_and_b, current_posts ,total_posts):
    # 计算当前的 P(A) * P(B)
    p_a_exception = p_a_and_b / p_b
    # 计算需要删除的帖子数量
    minus_posts = total_posts - current_posts / p_a_exception 
    print(f"需要删除 {minus_posts} 条帖子才能使 P(A ∩ B) = P(A) * P(B) 成立。")
    return minus_posts
calculate_minus_posts(P_B, P_A_and_B, len(valid_posts), len(posts_df))

2.2 删除文件中body字段无效的元组

In [ ]:
import pandas as pd
# 读取原始 CSV 文件
df = pd.read_csv(root + 'posts.csv')
# 统计原始数据的行数
original_count = len(df)
# 过滤出 'body' 字段不为 None 且不为空字符串的行
df_filtered = df[df['body'].notna() & (df['body'].str.strip() != '')]
# 统计保留的数据行数
filtered_count = len(df_filtered)
# 计算删除的元组数量
deleted_count = original_count - filtered_count
# 打印删除的元组数量
print(f"删除了 {deleted_count} 个元组。")
# 保存过滤后的数据为新 CSV 文件
df_filtered.to_csv(root + 'filtered_posts.csv', index=False)


2.3  从posts.csv文件中查找body属性中含有'trump'或'president‘或'DJT'的元组，这些元组的id必须和filtered_posts中的所有post的id不同，并将这些元组保存到新文件中，只保存下面字段 id:ID,:LABEL,comments,body,createdAt,creator,score,sensitive,upvotes,username,post,pcNum,entailment_probability,nli_label

In [ ]:
"""
从posts.csv文件中查找body属性中含有'trump'的1000个元组，这1000个元组的id必须和filtered_posts中的所有post的id不同，并将这些元组保存到新文件中，只保存下面字段
id:ID,:LABEL,comments,body,createdAt,creator,score,sensitive,upvotes,username,post,pcNum,entailment_probability,nli_label
"""
import pandas as pd

# 读取 posts.csv 和 filtered_posts.csv 文件
root = '/home/wangshuo/resource/datasets/parler_data/valid_posts/'
root_2 = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/'
posts_df = pd.read_csv(root + 'parler_posts_parler_data000000000022.csv')
filtered_posts_df = pd.read_csv(root_2 + 'posts_workload_1.csv')

# 获取 filtered_posts 中的所有 post 的 id
filtered_ids = filtered_posts_df['id:ID'].tolist()

# 定义关键词正则：查找 body 中包含 trump、president 或 DJT（不区分大小写）
regex_keywords = r'(trump|president|DJT)'

# 查找 body 字段中包含 'trump' 的元组，并且 id 不在 filtered_ids 中
filtered_trump_posts = posts_df[posts_df['body'].str.contains(regex_keywords, case=False, na=False)]
filtered_trump_posts = filtered_trump_posts[~filtered_trump_posts['id'].isin(filtered_ids)]

# 选取前 1000 个符合条件的元组
# trump_posts_to_save = filtered_trump_posts.head(1000)
trump_posts_to_save = filtered_trump_posts

# 只保留指定字段
columns_to_save = ['id', 'comments', 'body', 'createdAt', 'creator', 'score', 'sensitive', 'upvotes', 'username', 'post']
trump_posts_to_save = trump_posts_to_save[columns_to_save]

filename = 'posts_workload_1_supply_7.csv'
# 保存到新文件
trump_posts_to_save.to_csv(root_2 + filename, index=False)


print(f"已保存 {len(trump_posts_to_save)} 个符合条件的元组到 '{filename}'")


2.3.1 后处理文件，去除body字段中的换行符

In [ ]:
import pandas as pd

root_2 = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/valid_data/'
# 读取 CSV 文件
filename = 'user_lost.csv'
df = pd.read_csv(root_2 + filename)

# 打印前几行数据以确认成功读取
print(df.head())

# 去除 'body' 字段中的换行符
# df['body'] = df['body'].str.replace(r'[\n\r]', ' ', regex=True)
df['bio'] = df['bio'].str.replace(r'[\n\r]', ' ', regex=True)
# df['bodywithurls'] = df['bodywithurls'].str.replace(r'[\n\r]', ' ', regex=True)
# df['bio'] = df['bio'].str.replace(r'[\n\r]', ' ', regex=True)

# 保存处理后的数据到新文件
df.to_csv(root_2 + filename, index=False)

print(f"已去除 'body' 字段中的换行符，并保存为 '{filename}'")


2.4 对csv文件进行Qinfer处理打上标签

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm  # 导入 tqdm

# 设置模型文件路径
model_dir = '/home/wangshuo/resource/AIModels/NLP/bart-large-mnli'

# 加载模型和 tokenizer（从本地加载）
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

# 将模型转移到GPU（如果可用）
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
model.to(device)
print(device)

# 读取 CSV 文件中的数据
# data_dir = r'/home/wangshuo/resource/datasets/workload_20w'
data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3'
csv_file = data_dir + '/posts_workload_1_supply_7.csv'
print(csv_file)
df = pd.read_csv(csv_file)

# 获取所有的 "body" 字段内容
all_posts = df['body'].tolist()
print(f"数据加载完毕，共有 {len(all_posts)} 个帖子")

# 定义推理函数
def check_nli_batch(topic, batch_posts):
    """
    对每一批次的 post 进行 NLI 推理，判断其是否与给定的 topic 匹配
    """
    # 创建每个输入样本的 "topic + post" 对
    batch_inputs = [f"{post} [SEP] {topic}" for post in batch_posts]

    # 编码批量输入
    encoding = tokenizer(batch_inputs, padding=True, truncation=True, return_tensors="pt")
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # 模型推理
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        # 选择 "entailment" 和 "contradiction" 的 logits
        entail_contradiction_logits = logits[:, [0, 2]]

        # 计算 softmax 概率
        probs = entail_contradiction_logits.softmax(dim=1)

        # 获取 "entailment" 类别的概率（标签为真时的概率）
        prob_label_is_true = probs[:, 1].cpu().numpy()
        
        return prob_label_is_true

# 假设每批次处理 32 个文本
batch_size = 32
results = []

# topic = "Trump will win the presidency"
# topic = "Trump will make america great again"
topic = "Trump will win and save america"  # workload_1
# topic = "I support Trump"  # workload_1
# 使用 tqdm 显示进度条
for i in tqdm(range(0, len(all_posts), batch_size), desc="Processing Posts"):
    batch_posts = all_posts[i:i + batch_size]
    probabilities = check_nli_batch(topic, batch_posts)
    # 将每个样本的 "entailment" 概率添加到结果中
    results.extend(probabilities)


# 将结果添加到 DataFrame
# df['entailment_probability'] = results
df['workload1_probability'] = results
# 根据概率判断标签是否为 "entailment"（阈值为 0.5）
df['workload1_label'] = df['workload1_probability'].apply(lambda x: "workload1_entailment" if x and x > 0.5 else "workload1_not")
# df['nli_label'] = df['entailment_probability'].apply(lambda x: "bart_entailment" if x and x > 0.3 else "bart_not")

# 保存包含新列的 CSV 文件
output_csv_file = data_dir + '/posts_workload_1_supply_7.csv'
df.to_csv(output_csv_file, index=False)

print(f"推理结果已保存到 {output_csv_file}")


#### 2.5： posts_workload_1_supply_1.csv文件中workload1_label属性值为workload1_entailment的元组添加到posts_workload_1_extra中

In [ ]:
import pandas as pd
dataset_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/'
# 读取原始 CSV 文件
df = pd.read_csv( dataset_dir + 'posts_workload_1_supply_7.csv')

# 筛选出 workload1_label 为 workload1_entailment 的行
filtered_df = df[df['workload1_label'] == 'workload1_entailment']

# 将筛选结果追加到现有 CSV 文件
filtered_df.to_csv(dataset_dir + 'posts_workload_1_extra.csv', mode='a', header=False, index=False)


#### 2.5.1 comments_workload_1_extra.csv文件中只保留属性为:LABEL,id:ID,comments,body,createdAt,creator,score,sensitive,upvotes,username,controversy,parent,post,

In [ ]:
import pandas as pd
root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/'
# 读取 comments_workload_1_extra.csv 文件
df = pd.read_csv(root + 'comments_workload_1_extra.csv')

# 定义需要保留的列
columns_to_keep = ["id:ID", "comments", "body", "createdAt", "creator", 
                   "score", "sensitive", "upvotes", "username", "controversy", "parent", "post"]

# 筛选出指定的列
df_filtered = df[columns_to_keep]

# 保存筛选后的数据到新的 CSV 文件
df_filtered.to_csv(root + 'comments_workload_1_extra_filtered.csv', index=False)

print("已将文件只保留指定的属性，并保存为 'comments_workload_1_extra_filtered.csv'")


#### 2.6.1： 将posts_workload_1_extra.csv文件中的数据与posts_workload_1.csv文件中的数据合并到一起，保存到posts_combined.csv文件中

In [ ]:
import pandas as pd
root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/'
# filename = 'comments_workload_1_extra.csv'
filename = 'posts_workload_1_extra.csv'

# 读取 comments_workload_1_extra.csv 文件
extra_df = pd.read_csv(root + filename)

# 添加 ':LABEL' 列，并将其值设置为 'Comment'
# extra_df[':LABEL'] = 'Comment'
extra_df[':LABEL'] = 'Post'

# 读取 comments.csv 文件
# comments_df = pd.read_csv(root + 'comments_sorted_valid_4.csv')
comments_df = pd.read_csv(root + 'posts_workload_1.csv')

# 将 extra_df 中的数据添加到 comments_df 中
combined_df = pd.concat([comments_df, extra_df], ignore_index=True)

# 保存合并后的数据到新的 CSV 文件
# combined_df.to_csv(root + 'combined_comments.csv', index=False)
combined_df.to_csv(root + 'combined_posts.csv', index=False)

print("已将 'comments_workload_1_extra.csv' 中的数据添加到 'comments.csv' 中，并保存为 'combined_posts.csv'")


#### 2.6.2： 将comments_workload_1_extra.csv文件中的数据与comments_sorted_valid.csv文件中的数据合并到一起，保存到comments_combined.csv文件中

In [ ]:
import pandas as pd
root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/'
# 读取 A.csv 和 B.csv 文件
df_A = pd.read_csv(root + 'comments_workload_1_extra.csv')
df_B = pd.read_csv(root +'comments_sorted_valid_4.csv')

# 获取 B 文件的列顺序（最终输出的列顺序）
columns_B = list(df_B.columns)

# 对 A 文件：如果 B 中的某个列不存在，则补充缺失的列
for col in columns_B:
    if col not in df_A.columns:
        if col == ':LABEL':
            df_A[col] = "Comment"
        else:
            df_A[col] = ""  # 可根据需要填充为空字符串或 NaN

# 对 A 文件：保留 B 文件中存在的列（舍弃 A 中多余的列）
df_A = df_A[columns_B]

# 合并 B 文件和调整后的 A 文件（将 A 文件中的行追加到 B 文件后面）
df_combined = pd.concat([df_B, df_A], ignore_index=True)

# 保存合并后的数据到新的 CSV 文件中
output_file = root + 'comments_combined.csv'
df_combined.to_csv(output_file, index=False)

print(f"已将 A.csv 中的数据添加到 B.csv 中，并保存为 '{output_file}'")


 ### 下面是我文件夹/home/wangshuo/resource/datasets/parler_data/valid_commnets/comments下的某一个parler_comments_parler_data000000000000.csv的部分内容，其中post字段代表这个comment是某一个post的#评论，post字段和posts文件中某一个元组的id：ID一一对应，接下来我在文件夹/home/wangshuo/resource/datasets/parler_data/valid_commnets/comments下查找posts_workload_1_supply_3.csv文件中所有post的#commnets保存在comments_workload_1_extra.csv文件中，并保存每个posts查找到的对应comment的数量，记为pcNum保存回posts_workload_1_supply_3d文件中，记住文件夹下有好多csv文件都要一个个查询

In [ ]:
import pandas as pd
import glob
from tqdm import tqdm

root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/'
# 读取帖子数据
posts_df = pd.read_csv(root + 'posts_workload_1_extra.csv')
print(len(posts_df))
# 获取评论文件夹中所有 CSV 文件的路径
comments_files = glob.glob('/home/wangshuo/resource/datasets/parler_data/valid_commnets/comments/*.csv')

# 初始化一个空的列表，用于存储所有评论数据
all_comments = []

# 遍历每个评论文件，读取其内容，并将其添加到 all_comments 列表中
for file in tqdm(comments_files, desc="Processing comment files", unit="file"):
    df = pd.read_csv(file)
    all_comments.append(df)

# 将所有评论数据合并为一个 DataFrame
comments_df = pd.concat(all_comments, ignore_index=True)

# 筛选出与帖子相关的评论
filtered_comments_df = comments_df[comments_df['post'].isin(posts_df['id:ID'])]

# 将筛选后的评论保存到 comments_workload_1_extra.csv 文件中
filtered_comments_df.to_csv(root + 'comments_workload_1_extra.csv', index=False)

# 统计每个帖子的评论数量
comment_counts = filtered_comments_df.groupby('post').size().reset_index(name='pcNum')

# 将评论数量合并回原始帖子数据
posts_df = posts_df.merge(comment_counts, left_on='id:ID', right_on='post', how='left')

# 将结果保存回原始文件
posts_df.to_csv(root + 'posts_workload_1_extra.csv', index=False)

# 打印每个帖子的评论数量
for index, row in posts_df.iterrows():
    print(f"Post ID: {row['id:ID']}, Number of Comments: {row['pcNum']}")


In [ ]:
# 将筛选后的评论保存到 comments_workload_1_extra.csv 文件中
filtered_comments_df.to_csv(root + 'comments_workload_1_extra.csv', index=False)

# 统计每个帖子的评论数量
comment_counts = filtered_comments_df.groupby('post').size().reset_index(name='pcNum')

# 将评论数量合并回原始帖子数据
posts_df = posts_df.merge(comment_counts, left_on='id:ID', right_on='post', how='left')

# 将结果保存回原始文件
posts_df.to_csv(root + 'posts_workload_1_extra.csv', index=False)


3.1 统计每个概率段的数量

In [ ]:
import pandas as pd
root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/'
# 读取 comments.csv 文件
df = pd.read_csv(root + 'comments.csv')

# 定义阈值列表
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5,0.55, 0.6, 0.7, 0.8]

# 获取总行数
total_count = len(df)

# 统计每个阈值范围内的元组数量和比例
for threshold in thresholds:
    count = len(df[df['bart_entailment_probability'] > threshold])
    proportion = count / total_count
    print(f"bart_entailment_probability > {threshold}: {count} rows, {proportion:.2%} of total")


## 将这个文件元组按照bart_entailment_probability大小降序排序

In [ ]:
import pandas as pd

root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/'

input_file = 'combined_posts.csv'
output_file = 'posts_valid.csv'

# 读取 CSV 文件
df = pd.read_csv(root + input_file)

# 删除指定列
# df = df.drop(columns=['distilbart_entailment_probability', 'distilbart_nli_label'], errors='ignore')
df = df.drop(columns=['entailment_probability', 'nli_label' , 'id'], errors='ignore')

# 按照 'bart_entailment_probability' 列的值降序排序
df_sorted = df.sort_values(by='workload1_probability', ascending=False)

# 保存到新的 CSV 文件
df_sorted.to_csv(root + output_file, index=False)

print(f"已删除指定列，并按照 'bart_entailment_probability' 降序排序，保存为 {output_file}")


## 统计这个文件中body属性为空串或空的元组个数，并讲这些元组删除

In [ ]:
import pandas as pd

root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/'

# 读取 comments.csv 文件
df = pd.read_csv(root + 'comments_sorted.csv')

# 统计 'body' 字段为空或空字符串的元组数量
empty_body_count = df[df['body'].isnull() | (df['body'] == '') | (df['body'] == ' ')].shape[0]
print(f"共有 {empty_body_count} 个元组的 'body' 字段为空或空字符串。")

# 删除 'body' 字段为空或空字符串的元组
df_cleaned = df[~df['body'].isnull() & (df['body'] != '')]

# 保存清理后的数据到新的 CSV 文件
df_cleaned.to_csv('comments_cleaned.csv', index=False)
print("已删除 'body' 字段为空或空字符串的元组，并保存为 'comments_cleaned.csv'。")


## 统计comments_sorted_valid_2.csv文件中body属性含有'trump'（不区分大小写）单词的且nli_label=‘bart_entailment’元组数量有多少

In [ ]:
import pandas as pd

# 读取 comments_sorted_valid_2.csv 文件
root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/'
df = pd.read_csv(root + 'comments_sorted_valid_2.csv')

# 筛选出 'body' 字段包含 'trump' (不区分大小写) 且 'nli_label' 为 'bart_entailment' 的元组
filtered_df = df[df['body'].str.contains('trump', case=False, na=False) & (df['nli_label'] == 'bart_entailment')]

# 统计符合条件的元组数量
filtered_count = len(filtered_df)

# 打印符合条件的元组数量
print(f"符合条件的元组数量：{filtered_count}")


将comments_sorted_valid_2.csv文件中nli_label=‘bart_entailment’且body属性不含有'trump'（不区分大小写）单词的元组排到文件最前面，这些元组之间按照bart_entailment_probability大小降序排列，剩余的元组按照bart_entailment_probability大小降序排列，将排序好的元组保存到comments_sorted_valid_3.csv文件中

In [ ]:
import pandas as pd

root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/'

# 读取 comments_sorted_valid_2.csv 文件
df = pd.read_csv(root + 'comments_sorted_valid_2.csv')

# 筛选出符合条件的元组
filtered_df = df[
    (df['nli_label'] == 'bart_entailment') &
    (~df['body'].str.contains('trump', case=False, na=False)) &  # body 不含 'trump'
    (~df['body'].str.contains('maga', case=False, na=False)) &  # body 不含 'MAGa'
    (~df['body'].str.contains('president', case=False, na=False))  # body 不含 'president'
]

# 排序：首先按 bart_entailment_probability 降序排列
filtered_df = filtered_df.sort_values(by='bart_entailment_probability', ascending=False)

# 过滤后的数据和其他数据合并
remaining_df = df[~df.index.isin(filtered_df.index)]

# 其余数据也按 bart_entailment_probability 降序排列
remaining_df = remaining_df.sort_values(by='bart_entailment_probability', ascending=False)

# 将两个数据框合并，满足条件的数据在前
final_df = pd.concat([filtered_df, remaining_df])

# 保存结果到 comments_sorted_valid_4.csv 文件
final_df.to_csv(root + 'comments_sorted_valid_4.csv', index=False)

print("已将符合条件的元组排到最前，并按 bart_entailment_probability 排序，结果保存为 'comments_sorted_valid_4.csv'.")


In [ ]:
print(len(filtered_df))

## 读取下面文件的数据，删除body字段中只有一个单词或字符串长度小于4的元组

In [ ]:
import pandas as pd

root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/'

# 读取 comments_sorted.csv 文件
df = pd.read_csv(root + 'comments_sorted_valid.csv')

# 定义一个函数来判断 'body' 字段是否有效
def is_valid_body(text):
    # 去除首尾空白字符
    text = text.strip()
    # 判断是否为空
    if len(text) == 0:
        return False
    # 拆分文本为单词
    words = text.split()
    # 如果单词数量小于2或任何单词长度小于4，则返回 False
    if len(words) <= 2 :
        return False
    return True

# 统计原始 DataFrame 的行数
original_count = len(df)

# 应用该函数过滤有效的元组
df_filtered = df[df['body'].apply(is_valid_body)]

# 统计过滤后的 DataFrame 的行数
filtered_count = len(df_filtered)

# 计算删除的元组数量
deleted_count = original_count - filtered_count

# 保存过滤后的数据到新的 CSV 文件
df_filtered.to_csv(root + 'comments_sorted_valid_2.csv', index=False)

print(f"已删除 {deleted_count} 个元组，'body' 字段只有一个单词或单词长度小于4的元组，并保存为 'comments_sorted_valid.csv'。")


# 删除comments

In [ ]:
import pandas as pd
import random

# 读取评论数据和推文数据
# root = 'D:/softwareResource/datasets/workload4/'
root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/'
comments_df = pd.read_csv(root + 'combined_comments.csv')
posts_df = pd.read_csv(root + 'combined_posts.csv')  

# 1. 筛选出满足条件的评论
# 找到满足 nli_label = 'bart_not' 的评论
comments_bart_not = comments_df[comments_df['nli_label'] == 'bart_not']

# 找到这些评论关联的帖子，并筛选出其 nli_label 为 'not' 的帖子
valid_comments = comments_bart_not[comments_bart_not['post'].isin(posts_df[posts_df['nli_label'] == 'not']['id:ID'])]

# 2. 随机删除 66631 条评论
# 检查是否有足够的评论满足条件
num_comments_to_delete = 5000
if len(valid_comments) >= num_comments_to_delete:
    comments_to_delete = valid_comments.sample(n=num_comments_to_delete, random_state=42)
    
    # 3. 删除这些评论
    comments_del_df = comments_df[~comments_df['id:ID'].isin(comments_to_delete['id:ID'])]
    
    # 4. 保存删除后的数据到新的文件
    comments_del_df.to_csv(root + 'comments_del.csv', index=False)
    print(f"成功删除 {num_comments_to_delete} 条评论，保存到 comments_del.csv")
else:
    print(f"满足条件的评论不足 {num_comments_to_delete} 条，仅有 {len(valid_comments)} 条评论")



同理删除那些无效的post和user数据

In [ ]:
import pandas as pd
root = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/'
# 读取评论删除后的数据和原始帖子数据
comments_del_df = pd.read_csv(root + 'comments_del.csv')
posts_df = pd.read_csv(root+ 'posts.csv')

# 1. 获取在 comments_del.csv 中存在的所有评论关联的帖子 ID
valid_post_ids = comments_del_df['post'].unique()

# 2. 删除 posts.csv 中不在 valid_post_ids 中的帖子
posts_del_df = posts_df[posts_df['id:ID'].isin(valid_post_ids)]

# 3. 保存删除后的数据到新的文件 posts_del.csv
posts_del_df.to_csv(root + 'posts_del.csv', index=False)

print(f"删除后，保留的帖子数量为 {len(posts_del_df)} 条，已保存到 posts_del.csv")


In [ ]:
print(len(posts_df))

In [ ]:
print(root)

In [ ]:
import pandas as pd

# 读取数据
user_df = pd.read_csv(root + 'users.csv')
posts_del_df = pd.read_csv(root + 'posts_del.csv')
comments_del_df = pd.read_csv(root + 'comments_del.csv')

# 1. 获取 posts_del.csv 和 comments_del.csv 中所有的 username
valid_usernames = set(posts_del_df['username']).union(set(comments_del_df['username']))

# 2. 筛选 user.csv 中那些 username 出现在 valid_usernames 列表中的用户
valid_users_df = user_df[user_df['username'].isin(valid_usernames)]

# 3. 保存删除后的数据到新的文件 user_del.csv
valid_users_df.to_csv(root + 'users_del.csv', index=False)

print(f"删除后，保留的有效用户数量为 {len(valid_users_df)} 条，已保存到 user_del.csv")


In [ ]:
print(len(user_df))

   -----------------------------------------------------------------------------------------------------------    

In [ ]:
import pandas as pd
root = 'D:/softwareResource/datasets/IOGS/many_predicates/independent/dataset_1/'
# 读取 users.csv 文件
users_df = pd.read_csv(root + 'users.csv')

# 1. 筛选出 bio 同时包含 'family' 和 'maga' 的行
# 使用 str.contains() 进行不区分大小写的匹配
filtered_users = users_df[users_df['bio'].str.contains('family', case=False, na=False) &
                          users_df['bio'].str.contains('maga', case=False, na=False)]

# 2. 获取符合条件的用户数量
num_filtered_users = len(filtered_users)
# 输出符合条件的用户数量
print(f"符合条件的用户数量: {num_filtered_users}")

In [ ]:
import pandas as pd
dir = 'D:/softwareResource/datasets/IOGS/many_predicates/independent/dataset_1/'
# 读取 posts.csv 文件和 new_posts 数据
posts_df = pd.read_csv(dir + 'posts.csv')
new_posts_df = pd.read_csv(dir + 'new_posts.csv')

# 假设 posts.csv 和 new_posts.csv 中有相同的 id 字段，用于匹配
# 将 new_posts 中的 nli_labels 列更新到 posts 中
posts_df.set_index('id:ID', inplace=True)  # 设置 'id' 为索引，便于合并
new_posts_df.set_index('id:ID', inplace=True)  # 同样设置 'id' 为索引

# 使用 new_posts 的 nli_labels 更新 posts 的 nli_labels
posts_df['nli_label'] = new_posts_df['nli_label']

# 重置索引并保存更新后的 posts.csv 文件
posts_df.reset_index(inplace=True)
posts_df.to_csv(dir + 'posts_updated.csv', index=False)

print("nli_labels 更新完成，已保存到 'posts_updated.csv' 文件中")
